In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
import os

In [ ]:
def train_intent_model():
    print("Membaca dataset chat_dataset.csv...")
    dataset_path = "data/chat_dataset.csv"
    
    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None
        
    df = pd.read_csv(dataset_path)
    
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    print("Melatih model NLU (TF-IDF + SVM)...")
    model = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', probability=True))
    
    model.fit(X_train, y_train)
    
    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier.pkl")
    print("Model berhasil disimpan di models/intent_classifier.pkl\n")
    
    return model

In [ ]:
def predict_intent(chat_text):
    model_path = "models/intent_classifier.pkl"
    if not os.path.exists(model_path):
        model = train_intent_model()
    else:
        model = joblib.load(model_path)
    
    # Prediksi intent dari chat baru
    prediksi = model.predict([chat_text])[0]
    
    # Ambil nilai probabilitas/keyakinan model (dalam persentase)
    probabilitas = max(model.predict_proba([chat_text])[0]) * 100
    
    return prediksi, probabilitas

In [ ]:
train_intent_model()
    
print("--- SIMULASI TESTING AI 1 DI DALAM GAME ---")
test_chats = [
    "Jagain rumah gw pak pol, gw bayar mahal nih pake koin",
    "Lu curigaan mulu sama gw anjir, gw cuma warga biasa",
    "Woy si Budi dari tadi diem aja, fix dia ketuanya"
]

for chat in test_chats:
    intent, prob = predict_intent(chat)
    print(f"Chat Player: '{chat}'")
    print(f" > AI 1 Menebak: [{intent.upper()}] (Tingkat Keyakinan: {prob:.2f}%)\n")